# To annotate behaviour from timeseries


I am modifying this notebook so that it is more clear what it does.
It takes a Behaviour annotation in a similar format than this:
```
start,end,beh
1,14,Fwd
15,34,Rev
35,59,Coil
60,127,Fwd
128,143,Rev
144,166,Coil
167,178,Fwd
179,211,Rev
212,232,Coil
233,268,Fwd
```
And converts them into a timeseries where everyvolume is annotated.
```
Volume,Description,Annotation
0,Fwd,0.0
1,Fwd,0.0
2,Fwd,0.0
3,Fwd,0.0
4,Fwd,0.0
..

12,Fwd,0.0
13,Fwd,0.0
14,Rev,1.0
15,Rev,1.0
16,Rev,1.0
..
33,Rev,1.0
34,Coil,2.0
35,Coil,2.0
36,Coil,2.0

..
```
And so on

## Steps
### 1. Subsample your behaviour image
If you have more than one behaviour image per datapoint (e.g. wbfm confocal imaging) subsample the image.

If each volume had 22 planes you need to get Use slice keeper with Increment 22 (Fiji>Image>Stacks>Tools>Slice Keeper)

### 2. Manually Annotate Behaviour
(In the future this will be done automatically with PCA cross product Rev annotation)

An easy way to do it is to annotate the starting frame, last frame and behaviour
    
    start,end,beh
    1,30,Fwd
    31,40,Rev
    
   etc.
   
save it in a csv file in the parent directory of the recording as beh_annotation.csv
### 3. Run the code
The code will do what is described in the previous cell.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import tifffile as tiff
from natsort import natsorted

In [66]:
# load the manually annotated behaviours
path='/Volumes/scratch/ulises/wbfm/20211217/data/worm9/beh_annotation.csv'
df=pd.read_csv(path)

In [67]:
df

,start,end,beh
0,1,14,Fwd
1,15,34,Rev
2,35,59,Coil
3,60,127,Fwd
4,128,143,Rev
5,144,166,Coil
6,167,178,Fwd
7,179,211,Rev
8,212,232,Coil
9,233,268,Fwd


In [68]:
# get the max number of volumes (or frames)
number_of_volumes=df['end'].max()

In [69]:
# create a new pandas dataframe
timeline=pd.DataFrame(index=np.arange(number_of_volumes))

In [70]:
# this dictionary links Strings of behaviours to Annotations in numbers
beh_dict={'Fwd': 0,
          'Rev': 1,
          'Coil': 2}

In [73]:
# iterate over the dataframe and save the data in the timeline dataframe
for row,label in df.iterrows():
    # print(row)
    # print(label)
    # to have cleaner code i specify these variables. -1 is because the first annotation took the fiji indexin which starts with 1
    start=label['start']-1
    end=label['end']-1
    beh=label['beh']
    timeline.loc[start:end,'Description']=beh
    timeline.loc[start:end,'Annotation']=int(beh_dict[beh])

# optional: save index name as 'Volume'
timeline.index.name = 'Volume'

In [74]:
# visualize the result
timeline

,Description,Annotation
Volume,,
0,Fwd,0.0
1,Fwd,0.0
2,Fwd,0.0
3,Fwd,0.0
4,Fwd,0.0
...,...,...
2397,Fwd,0.0
2398,Fwd,0.0
2399,Fwd,0.0


In [75]:
# save it
output_path='/Volumes/scratch/ulises/wbfm/20211217/data/worm9/beh_annotation_timeseries.csv'
timeline.to_csv(output_path)